# Monte Carlo — So sánh 4 Filter Tracking

**Framework tham khảo:** Lee et al. (2021), *IEEE Access* 9:30993–31009 — "Performance Verification of a Target Tracking System with a Laser Rangefinder".

**Mục tiêu:** Chạy N=100 lần Monte Carlo trên 5 scenario, so sánh:
1. **ABF (α-β)** — filter chính của đề tài (PRESET AB-B: α=0.30, β=α²/(2−α)=0.0529)
2. **ABGF (α-β-γ)** — mở rộng cho accelerating target (3-state CA)
3. **CV-KF** — Kalman 2-state constant-velocity
4. **CA-KF** — Kalman 3-state constant-acceleration

**Bien luận:** Nếu ABF cho position RMSE ~ Kalman variants, chọn ABF hợp lý cho hệ nhúng ESP32 vì cost/performance ratio tối ưu.

**Ghi chú semantic:**
- Notebook này dùng filter **thuần** (không có FSM TrackerCore). Gate reject → predict-only; sau `max_reject` liên tiếp → reinit.
- Constants đã sync với firmware v1.2.3-UX-v2 PresetTable AB-B.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 10

np.random.seed(42)  # top-level seed cho reproducibility

## 2. Filter definitions

### 2.1 Alpha-Beta (ABF)
2-state CV, preset **AB-B** của firmware. β theo Benedict–Bordner *gain relationship* — **KHÔNG** phải "critically damped" (Benedict & Bordner, 1962).

In [ ]:
class AlphaBetaFilter:
    """Alpha-Beta filter (2-state CV) - preset AB-B cua firmware v1.2.3-UX-v2.
    alpha = 0.30, beta = alpha^2/(2-alpha) = 0.0529 (Benedict-Bordner gain relationship).
    """
    NAME = "ABF (a-b)"

    def __init__(self, alpha=0.30, beta=0.0529, dt=0.4, gate=4.0,
                 max_reject=5, decay=0.85):
        # gate=4.0 khop PresetTable AB-B (khong phai 10m default cu)
        self.alpha = alpha
        self.beta = beta
        self.dt = dt
        self.gate = gate
        self.max_reject = max_reject
        self.decay = decay
        self.reset()

    def reset(self):
        self.x = None
        self.v = 0.0
        self.reject_count = 0

    def update(self, z, valid=True):
        if not valid:
            if self.x is not None:
                self.v *= self.decay          # B4 velocity decay
                self.x += self.v * self.dt
            return self.x
        if self.x is None:
            self.x = z; self.v = 0.0
            return z
        x_pred = self.x + self.v * self.dt
        r = z - x_pred
        if abs(r) > self.gate:
            self.reject_count += 1
            if self.reject_count >= self.max_reject:
                self.x = z; self.v = 0.0; self.reject_count = 0
                return z
            self.x = x_pred                   # B2 predict-only
            return x_pred
        self.reject_count = 0
        self.x = x_pred + self.alpha * r
        self.v = self.v + (self.beta / self.dt) * r
        return self.x

### 2.2 Alpha-Beta-Gamma (ABGF) — 3-state CA

In [ ]:
class AlphaBetaGammaFilter:
    NAME = "ABGF (a-b-g)"
    def __init__(self, alpha=0.45, beta=0.15, gamma=0.01, dt=0.4,
                 gate=4.0, max_reject=5, decay=0.85):
        self.alpha, self.beta, self.gamma = alpha, beta, gamma
        self.dt, self.gate = dt, gate
        self.max_reject, self.decay = max_reject, decay
        self.reset()
    def reset(self):
        self.x = None; self.v = 0.0; self.a = 0.0; self.reject_count = 0
    def update(self, z, valid=True):
        if not valid:
            if self.x is not None:
                self.v *= self.decay
                self.x += self.v * self.dt + 0.5 * self.a * self.dt**2
            return self.x
        if self.x is None:
            self.x = z; return z
        x_pred = self.x + self.v * self.dt + 0.5 * self.a * self.dt**2
        r = z - x_pred
        if abs(r) > self.gate:
            self.reject_count += 1
            if self.reject_count >= self.max_reject:
                self.x = z; self.v = 0.0; self.a = 0.0; self.reject_count = 0
                return z
            self.x = x_pred
            return x_pred
        self.reject_count = 0
        self.x = x_pred + self.alpha * r
        self.v = self.v + (self.beta / self.dt) * r
        self.a = self.a + (2 * self.gamma / (self.dt**2)) * r
        return self.x

### 2.3 Kalman CV (2-state)

In [ ]:
class KalmanCV:
    NAME = "CV-KF"
    def __init__(self, dt=0.4, sigma_z=0.2, sigma_a=1.0, gate_sigma=3.0, max_reject=5):
        self.dt, self.sigma_z = dt, sigma_z
        self.F = np.array([[1.0, dt], [0.0, 1.0]])
        self.H = np.array([[1.0, 0.0]])
        self.R = sigma_z**2
        self.Q = sigma_a**2 * np.array([[dt**4/4, dt**3/2],[dt**3/2, dt**2]])
        self.gate_sigma, self.max_reject = gate_sigma, max_reject
        self.reset()
    def reset(self):
        self.x = None; self.P = np.eye(2)*100.0; self.reject_count = 0
    def update(self, z, valid=True):
        if not valid:
            if self.x is not None:
                self.x = self.F @ self.x
                self.P = self.F @ self.P @ self.F.T + self.Q
            return self.x[0] if self.x is not None else None
        if self.x is None:
            self.x = np.array([z, 0.0]); self.P = np.diag([self.sigma_z**2, 10.0])
            return z
        x_pred = self.F @ self.x
        P_pred = self.F @ self.P @ self.F.T + self.Q
        y = z - float(self.H @ x_pred)
        S = float(self.H @ P_pred @ self.H.T) + self.R
        sqrt_S = np.sqrt(max(S, 1e-9))
        if abs(y) > self.gate_sigma * sqrt_S:
            self.reject_count += 1
            if self.reject_count >= self.max_reject:
                self.x = np.array([z, 0.0]); self.P = np.diag([self.sigma_z**2, 10.0])
                self.reject_count = 0
                return z
            self.x = x_pred; self.P = P_pred
            return float(x_pred[0])
        self.reject_count = 0
        K = (P_pred @ self.H.T).flatten() / S
        self.x = x_pred + K * y
        self.P = (np.eye(2) - np.outer(K, self.H.flatten())) @ P_pred
        return float(self.x[0])

### 2.4 Kalman CA (3-state)

In [ ]:
class KalmanCA:
    NAME = "CA-KF"
    def __init__(self, dt=0.4, sigma_z=0.2, sigma_j=1.0, gate_sigma=3.0, max_reject=5):
        self.dt, self.sigma_z = dt, sigma_z
        self.F = np.array([[1.0, dt, dt**2/2],[0.0, 1.0, dt],[0.0, 0.0, 1.0]])
        self.H = np.array([[1.0, 0.0, 0.0]])
        self.R = sigma_z**2
        self.Q = sigma_j**2 * np.array([
            [dt**5/20, dt**4/8,  dt**3/6],
            [dt**4/8,  dt**3/3,  dt**2/2],
            [dt**3/6,  dt**2/2,  dt]
        ])
        self.gate_sigma, self.max_reject = gate_sigma, max_reject
        self.reset()
    def reset(self):
        self.x = None; self.P = np.eye(3)*100.0; self.reject_count = 0
    def update(self, z, valid=True):
        if not valid:
            if self.x is not None:
                self.x = self.F @ self.x
                self.P = self.F @ self.P @ self.F.T + self.Q
            return self.x[0] if self.x is not None else None
        if self.x is None:
            self.x = np.array([z, 0.0, 0.0]); self.P = np.diag([self.sigma_z**2, 10.0, 1.0])
            return z
        x_pred = self.F @ self.x
        P_pred = self.F @ self.P @ self.F.T + self.Q
        y = z - float(self.H @ x_pred)
        S = float(self.H @ P_pred @ self.H.T) + self.R
        sqrt_S = np.sqrt(max(S, 1e-9))
        if abs(y) > self.gate_sigma * sqrt_S:
            self.reject_count += 1
            if self.reject_count >= self.max_reject:
                self.x = np.array([z, 0.0, 0.0]); self.P = np.diag([self.sigma_z**2, 10.0, 1.0])
                self.reject_count = 0
                return z
            self.x = x_pred; self.P = P_pred
            return float(x_pred[0])
        self.reject_count = 0
        K = (P_pred @ self.H.T).flatten() / S
        self.x = x_pred + K * y
        self.P = (np.eye(3) - np.outer(K, self.H.flatten())) @ P_pred
        return float(self.x[0])

## 3. Scenarios

5 kịch bản mô phỏng, mỗi kịch bản chạy N lần với seed khác nhau:

| Scenario | Motion | Mục đích |
|---|---|---|
| Static @180m | R(t) = 180 | Baseline jitter |
| Outbound 1.5 m/s | R(t) = 50 + 1.5t | CV di ra xa |
| Inbound 2 m/s | R(t) = 200 − 2t | CV lại gần |
| Step change (100→250) | switch tại t=25s | Target-switch reinit |
| Accelerating 0.15 m/s² | R(t) = 100 + 0.5t + 0.075t² | Test ABGF advantage |

In [ ]:
def sc_static(t, R0=180.0):        return np.full_like(t, R0)
def sc_outbound(t, R0=50.0, v=1.5): return R0 + v * t
def sc_inbound(t, R0=200.0, v=-2.0):return R0 + v * t
def sc_step_change(t, R1=100.0, R2=250.0, t_switch=25.0):
    return np.where(t < t_switch, R1, R2)
def sc_accelerating(t, R0=100.0, v0=0.5, a=0.15):
    return R0 + v0 * t + 0.5 * a * t * t

SCENARIOS = {
    "Static @180m":              (sc_static,       {"R0": 180.0}),
    "Outbound 1.5 m/s":          (sc_outbound,     {"R0": 50.0, "v": 1.5}),
    "Inbound 2 m/s":             (sc_inbound,      {"R0": 200.0, "v": -2.0}),
    "Step change (100 -> 250)":  (sc_step_change,  {}),
    "Accelerating 0.15 m/s^2":   (sc_accelerating, {}),
}

## 4. Noise model — mimic TC22 measurement

- **Gaussian σ = 0.2m** (đo từ bench 180m tripod)
- **Miss-hit 5%** — sample invalid (mất tín hiệu)
- **Outlier 2%** — sample valid nhưng lệch ±5m (phản xạ sai)

In [ ]:
def apply_measurement_noise(R_true, sigma=0.2, miss_hit_rate=0.05,
                            outlier_rate=0.02, outlier_amp=5.0, seed=42):
    rng = np.random.RandomState(seed)
    n = len(R_true)
    z = R_true + rng.normal(0, sigma, n)
    valid = rng.random(n) > miss_hit_rate
    outlier_mask = (rng.random(n) < outlier_rate) & valid
    z[outlier_mask] += rng.choice([-1, 1], outlier_mask.sum()) * outlier_amp
    return z, valid

## 5. Runner + metrics

Bỏ **25 samples warmup** (10s @ 2.5fps) trước khi tính RMSE để loại transient hội tụ.

In [ ]:
def run_filter(filter_obj, z_meas, valid, R_true, warmup=25):
    filter_obj.reset()
    est = np.full(len(z_meas), np.nan)
    for i, (zi, vi) in enumerate(zip(z_meas, valid)):
        result = filter_obj.update(float(zi), valid=bool(vi))
        if result is not None:
            est[i] = result
    est_w = est[warmup:]; truth_w = R_true[warmup:]
    mask = ~np.isnan(est_w)
    if mask.sum() < 10:
        return {"pos_rmse": np.nan, "avail": 0.0, "est": est}
    err = est_w[mask] - truth_w[mask]
    return {
        "pos_rmse": float(np.sqrt(np.mean(err**2))),
        "avail":    float(mask.mean()),
        "est":      est,
    }

def make_filters():
    """Factory tao 4 filter voi dt=0.4 (2.5 fps TC22), gate = AB-B PresetTable."""
    return {
        "ABF (a-b)":    AlphaBetaFilter(alpha=0.30, beta=0.0529, dt=0.4, gate=4.0),
        "ABGF (a-b-g)": AlphaBetaGammaFilter(alpha=0.45, beta=0.15, gamma=0.01,
                                             dt=0.4, gate=4.0),
        "CV-KF":        KalmanCV(dt=0.4, sigma_z=0.2, sigma_a=0.5),
        "CA-KF":        KalmanCA(dt=0.4, sigma_z=0.2, sigma_j=0.5),
    }

def monte_carlo_scenario(scenario_fn, scenario_kwargs, filter_factory,
                         N=100, dt=0.4, T=60.0):
    t = np.arange(0, T, dt)
    R_true = scenario_fn(t, **scenario_kwargs)
    per_filter_rmse = {name: [] for name in filter_factory().keys()}
    for seed in range(N):
        z_meas, valid = apply_measurement_noise(R_true, seed=seed)
        filters = filter_factory()
        for name, filt in filters.items():
            r = run_filter(filt, z_meas, valid, R_true)
            per_filter_rmse[name].append(r["pos_rmse"])
    summary = {}
    for name, rmses in per_filter_rmse.items():
        arr = np.array([x for x in rmses if not np.isnan(x)])
        summary[name] = {
            "mean_rmse":   float(arr.mean()),
            "std_rmse":    float(arr.std()),
            "median_rmse": float(np.median(arr)),
            "n_runs":      int(len(arr)),
        }
    return summary

## 6. Chạy Monte Carlo

Điều chỉnh `N_RUNS` nếu chạy chậm (100 runs × 5 scenarios × 4 filters ≈ 15s trên laptop hiện đại).

In [ ]:
N_RUNS = 100
DT = 0.4
T = 60.0

all_results = {}
for sc_name, (sc_fn, sc_kw) in SCENARIOS.items():
    summary = monte_carlo_scenario(sc_fn, sc_kw, make_filters, N=N_RUNS, dt=DT, T=T)
    all_results[sc_name] = summary
    print(f'\n--- {sc_name} ---')
    print(f'{"Filter":<15} {"Mean RMSE (m)":>15} {"Std (m)":>10} {"Median (m)":>12}')
    print('-' * 55)
    for name, s in summary.items():
        print(f'{name:<15} {s["mean_rmse"]:>15.4f} {s["std_rmse"]:>10.4f} {s["median_rmse"]:>12.4f}')

## 7. Xuất bảng kết quả CSV

In [ ]:
rows = []
for sc, filters in all_results.items():
    for f, s in filters.items():
        rows.append({
            "scenario":      sc,
            "filter":        f,
            "mean_rmse_m":   s["mean_rmse"],
            "std_rmse_m":    s["std_rmse"],
            "median_rmse_m": s["median_rmse"],
            "n_runs":        s["n_runs"],
        })
df_results = pd.DataFrame(rows)
df_results.to_csv("monte_carlo_results.csv", index=False)
print(f"[SAVED] monte_carlo_results.csv ({len(df_results)} rows)")
display(df_results)

## 8. Plot 1 — Bar chart RMSE mỗi scenario

In [ ]:
filter_names = list(make_filters().keys())
scenario_names = list(SCENARIOS.keys())
n_f = len(filter_names)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 1]})
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
width = 0.2

non_step = [sc for sc in scenario_names if "Step" not in sc]
x = np.arange(len(non_step))
for i, fname in enumerate(filter_names):
    means = [all_results[sc][fname]["mean_rmse"] for sc in non_step]
    stds = [all_results[sc][fname]["std_rmse"] for sc in non_step]
    ax1.bar(x + i * width, means, width, yerr=stds, capsize=3,
            label=fname, color=colors[i], alpha=0.85)
ax1.set_xticks(x + width * (n_f - 1) / 2)
ax1.set_xticklabels(non_step, rotation=12, ha="right", fontsize=9)
ax1.set_ylabel("Mean position RMSE (m)")
ax1.set_title("Steady-state scenarios (CV / accelerating)")
ax1.legend(fontsize=9)
ax1.grid(axis="y", alpha=0.3)

step_scen = [sc for sc in scenario_names if "Step" in sc][0]
xs = np.arange(1)
for i, fname in enumerate(filter_names):
    m = all_results[step_scen][fname]["mean_rmse"]
    s = all_results[step_scen][fname]["std_rmse"]
    ax2.bar(xs + i * width, [m], width, yerr=[s], capsize=3, color=colors[i], alpha=0.85)
ax2.set_xticks([xs[0] + width * (n_f - 1) / 2])
ax2.set_xticklabels([step_scen], fontsize=9)
ax2.set_ylabel("Mean position RMSE (m)")
ax2.set_title("Step change (transient dominated)")
ax2.grid(axis="y", alpha=0.3)

fig.suptitle(f"Monte Carlo (N={N_RUNS}): Filter position RMSE — framework Lee et al. (2021)",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("monte_carlo_pos_rmse.png", dpi=140, bbox_inches="tight")
plt.show()

## 9. Plot 2 — Example run mỗi scenario (seed=0)

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(12, 9), sharex=False)
axs_flat = axs.flatten()

for idx, (sc_name, (sc_fn, sc_kw)) in enumerate(SCENARIOS.items()):
    if idx >= 6: break
    ax = axs_flat[idx]
    t = np.arange(0, T, DT)
    R_true = sc_fn(t, **sc_kw)
    z_meas, valid = apply_measurement_noise(R_true, seed=0)

    ax.plot(t, R_true, "k-", lw=1.5, label="Truth", alpha=0.7)
    ax.scatter(t[valid], z_meas[valid], s=8, color="gray", alpha=0.5, label="Measurement")

    filters = make_filters()
    for fname, filt in filters.items():
        r = run_filter(filt, z_meas, valid, R_true)
        ax.plot(t, r["est"], lw=1.4, label=fname, alpha=0.9)

    ax.set_title(sc_name, fontsize=10)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Range (m)")
    ax.grid(alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=7, loc="best")

for j in range(len(SCENARIOS), 6):
    axs_flat[j].axis("off")

plt.suptitle("Example run per scenario (seed=0)", y=1.0, fontsize=11)
plt.tight_layout()
plt.savefig("monte_carlo_scenarios.png", dpi=140)
plt.show()

## 10. Diễn giải cho báo cáo Chương 6.6

**Kỳ vọng theo lý thuyết (Lee 2021, Bar-Shalom):**
- **Static / CV scenarios:** cả 4 filter cho RMSE tương đương (~σ_z = 0.2m). ABF không thua Kalman.
- **Accelerating:** ABGF/CA-KF nhỉnh hơn nhờ có state acceleration; ABF/CV-KF có lag bias.
- **Step change:** cả 4 đều depend vào reinit-on-switch. Filter nào reinit nhanh (max_reject ngắn) sẽ win transient period.

**Bien luận chọn ABF cho ESP32:**
1. RMSE ≈ Kalman variants trên CV scenarios (chiếm >90% use case bore-sight tripod)
2. Cost: 2 phép nhân/mẫu vs Kalman ≥ 10 phép nhân/mẫu + inverse ma trận (không có FPU 32-bit)
3. Không cần tuning Q/R noise covariance (chỉ cần α/β theo Benedict–Bordner)
4. Đủ cho spec δR ≤ 0.5m ở R ≤ 500m
5. **Trade-off:** không adaptive theo maneuver — chấp nhận được vì target thesis là quan sát tĩnh

**Cần verify với real data (Chương 6.3–6.5):** MC dùng noise model đơn giản, real TC22 có thể có clutter và range walk mà MC không capture. Kết quả on-device DT1/DT3 mới là ground truth cuối cùng.